# H&E ↔ PSR WSI Registration — VALIS

Registers H&E and PSR stained mouse liver WSIs using **VALIS**, which uses
SIFT feature matching for robust multi-stain registration.

**Advantages over wsireg/Elastix for this use case:**
- Always registers on thumbnails (`MAX_PROC_SIZE`) → no image-size hangs
- Feature-based (SIFT) → more robust across H&E ↔ PSR than intensity-based
- Reads flat `.tif` files directly — no pyramid conversion needed

**Setup (do this before running):**
```bash
# macOS
brew install vips openslide
pip install valis-wsi openslide-python tifffile matplotlib

# Ubuntu
sudo apt install libvips-dev libopenslide-dev
pip install valis-wsi openslide-python tifffile matplotlib
```

**Expected input — H&E and PSR in separate directories:**
```
HE_DIR/          PSR_DIR/
├── mouse01.tif  ├── mouse01.tif
├── mouse02.tif  └── mouse02.tif
└── mouse03.tif
```
Files are paired by matching stems. Stain keywords in filenames are stripped before matching.

## 1 — Configuration

In [ ]:
# ── Edit these before running ─────────────────────────────────────────────────

HE_DIR     = "/path/to/HE/files"       # folder containing H&E .tif files
PSR_DIR    = "/path/to/PSR/files"      # folder containing PSR .tif files
OUTPUT_DIR = "./valis_output"           # registered results go here

# Optional stain keywords in filenames (case-insensitive).
# Set to "" if stems already match directly (e.g. mouse01.tif / mouse01.tif).
HE_KEYWORD  = "HE"
PSR_KEYWORD = "PSR"

# ── VALIS registration settings ───────────────────────────────────────────────

# Max image dimension (px) used for registration computation.
# VALIS downsamples to this automatically — larger = more accurate but slower.
# Default in VALIS 1.x is 512. 850 gives better accuracy for tissue sections.
MAX_PROC_SIZE = 850

# Non-rigid optical-flow step after rigid+affine.
# Set True only if QC overlay shows local warping after rigid+affine.
NON_RIGID = False

## 2 — Imports

In [ ]:
import os
import re
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from valis import registration

## 3 — Discover and pair files

In [ ]:
def collect_tifs(directory: str) -> list:
    d = Path(directory)
    return sorted(d.glob("*.tif")) + sorted(d.glob("*.tiff")) + sorted(d.glob("*.svs"))


def strip_keyword(stem: str, keyword: str) -> str:
    if not keyword:
        return stem
    return re.sub(
        rf"[_\-.]?{re.escape(keyword)}[_\-.]?", "", stem, flags=re.IGNORECASE
    ).strip("_-.")


he_files  = collect_tifs(HE_DIR)
psr_files = collect_tifs(PSR_DIR)

he_map  = {strip_keyword(f.stem, HE_KEYWORD):  f for f in he_files}
psr_map = {strip_keyword(f.stem, PSR_KEYWORD): f for f in psr_files}

common_keys = sorted(set(he_map) & set(psr_map))
pairs = [(he_map[k], psr_map[k]) for k in common_keys]

print(f"H&E  directory : {HE_DIR}  ({len(he_files)} files)")
print(f"PSR  directory : {PSR_DIR}  ({len(psr_files)} files)")
print(f"Matched pairs  : {len(pairs)}\n")
for he, psr in pairs:
    print(f"  HE : {he.name}")
    print(f"  PSR: {psr.name}")
    print()

unmatched_he  = set(he_map)  - set(psr_map)
unmatched_psr = set(psr_map) - set(he_map)
if unmatched_he:
    print(f"Warning — unmatched H&E : {unmatched_he}")
if unmatched_psr:
    print(f"Warning — unmatched PSR : {unmatched_psr}")

## 4 — Register each pair

In [ ]:
def register_pair(he_path: Path, psr_path: Path, out_dir: Path,
                  max_proc_size: int, non_rigid: bool) -> Path:
    pair_name = strip_keyword(he_path.stem, HE_KEYWORD)
    pair_out  = out_dir / pair_name

    if pair_out.exists() and any(pair_out.rglob('*.ome.tiff')):
        print(f'  [skip] {pair_name} already registered')
        return pair_out

    tmp_dir  = out_dir / f'.tmp_{pair_name}'
    tmp_dir.mkdir(parents=True, exist_ok=True)

    he_link  = tmp_dir / he_path.name
    psr_link = tmp_dir / psr_path.name
    for link, target in [(he_link, he_path), (psr_link, psr_path)]:
        if not link.exists():
            os.symlink(target.resolve(), link)

    non_rigid_cls = registration.OptFlowRegistrar if non_rigid else None

    registrar = registration.Valis(
        src_dir=str(tmp_dir),
        dst_dir=str(pair_out),
        reference_img_f=he_path.name,
        max_processed_image_dim_px=max_proc_size,
        non_rigid_registrar_cls=non_rigid_cls,
        align_to_reference=True,
        imgs_ordered=False,
    )

    rigid_registrar, non_rigid_registrar, error_df = registrar.register()

    if error_df is not None:
        print('  Registration error summary:')
        print(error_df[['name', 'original_D', 'registered_D']].to_string(index=False))
    else:
        print('  Warning: registration did not return error metrics.')

    reg_out = pair_out / 'registered'
    reg_out.mkdir(parents=True, exist_ok=True)   # parents=True in case pair_out wasn't created

    # crop='none' avoids a mask_dict crash that occurs with align_to_reference=True
    registrar.warp_and_save_slides(dst_dir=str(reg_out), crop='none')

    shutil.rmtree(tmp_dir, ignore_errors=True)
    return pair_out


out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

registered_results = []

print(f"MAX_PROC_SIZE : {MAX_PROC_SIZE} px")
print(f"NON_RIGID     : {NON_RIGID}\n")

for i, (he, psr) in enumerate(pairs):
    print(f"\n[{i+1}/{len(pairs)}] {strip_keyword(he.stem, HE_KEYWORD)}")
    result = register_pair(he, psr, out_dir, MAX_PROC_SIZE, NON_RIGID)
    registered_results.append((he, psr, result))

print("\nAll registrations complete.")

## 5 — QC overlays

In [ ]:
def read_thumb(path: Path, max_dim: int = 1000) -> np.ndarray:
    """Read a thumbnail from any image format via PIL fallback."""
    try:
        import tifffile
        with tifffile.TiffFile(str(path)) as tif:
            series = tif.series[0]
            # Pick the smallest level that's still >= max_dim on the long side
            for lvl in reversed(range(len(series.levels))):
                arr = series.levels[lvl].asarray()
                if arr.ndim == 3 and arr.shape[0] in (1, 3, 4):
                    arr = np.moveaxis(arr, 0, -1)
                h, w = arr.shape[:2]
                if max(h, w) >= max_dim or lvl == 0:
                    break
            arr = arr[..., :3].astype(np.uint8)
    except Exception:
        arr = np.array(Image.open(str(path)).convert("RGB"))

    # Resize to max_dim on long side
    h, w = arr.shape[:2]
    scale = max_dim / max(h, w)
    new_h, new_w = int(h * scale), int(w * scale)
    return np.array(Image.fromarray(arr).resize((new_w, new_h), Image.LANCZOS))


def checkerboard(a: np.ndarray, b: np.ndarray, n: int = 8) -> np.ndarray:
    h, w   = a.shape[:2]
    th, tw = h // n, w // n
    board  = a.copy()
    for r in range(n):
        for c in range(n):
            if (r + c) % 2 == 0:
                board[r*th:(r+1)*th, c*tw:(c+1)*tw] = b[r*th:(r+1)*th, c*tw:(c+1)*tw]
    return board


qc_dir = out_dir / "qc"
qc_dir.mkdir(exist_ok=True)

for he, psr, pair_out in registered_results:
    pair_name = strip_keyword(he.stem, HE_KEYWORD)

    # Find the registered PSR in the output folder
    reg_files = list((pair_out / "registered").glob("*PSR*")) + \
                list((pair_out / "registered").glob("*psr*"))
    if not reg_files:
        reg_files = list((pair_out / "registered").glob("*"))

    he_arr      = read_thumb(he)
    psr_arr     = read_thumb(psr)
    reg_psr_arr = read_thumb(reg_files[0]) if reg_files else psr_arr

    # Resize all to HE thumbnail dimensions
    h, w = he_arr.shape[:2]
    def _rsz(arr):
        return np.array(Image.fromarray(arr).resize((w, h), Image.LANCZOS))
    psr_arr     = _rsz(psr_arr)
    reg_psr_arr = _rsz(reg_psr_arr)
    board       = checkerboard(he_arr, reg_psr_arr)

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(pair_name, fontsize=13, fontweight="bold")
    for ax, img, title in zip(
        axes,
        [he_arr, psr_arr, reg_psr_arr, board],
        ["H&E (fixed)", "PSR (original)", "PSR (registered)", "Checkerboard"],
    ):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    out_png = qc_dir / f"{pair_name}_qc.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png}")

## 6 — Summary

In [ ]:
print("═" * 60)
print(f"  Pairs processed  : {len(registered_results)}")
print(f"  Registered WSIs  : {out_dir}/<sample>/registered/")
print(f"  QC overlays      : {qc_dir}")
print("═" * 60)
print()
for he, psr, pair_out in registered_results:
    name = strip_keyword(he.stem, HE_KEYWORD)
    reg_files = list((pair_out / "registered").glob("*")) if (pair_out / "registered").exists() else []
    for f in reg_files:
        print(f"  {name} → {f.name}")